# The final selector: v3 with EmbeddingGemma 2
**Ariel Lax** | Corpus2GeoRAG / CollectCorpus

Chosen after CompareEmbeddings.ipynb: the same accuracy as E5, more related pages accepted, faster.
This notebook
1. trains the final selector (the 3,009 v3 paragraphs, Gemma vectors) and saves it to `MyDrive/selector/model_gemma`;
2. scores the 3,551 new pages found by the BFS (one link away from the corpus) with **Gemma and with E5 v3**,
   so the final corpus can be compared with the E5 version.

### Before running
1. Nothing new to upload: everything it needs is already in `MyDrive/selector`.
2. Runtime → Change runtime type → **T4 GPU**. Runtime → **Run all**, and approve the Drive connection.
   About an hour; if it disconnects, Run all again and it continues from the saved chunks.
3. At the end, download from `MyDrive/selector` to `CollectCorpus/data/selector/v3/`:
   **`gemma_candidate_scores.csv`** and the folder **`model_gemma`** (2 small files).

## Setup

In [ ]:
# EmbeddingGemma 2 needs recent versions (run on a fresh runtime, before anything imports transformers)
%pip install -q -U sentence-transformers transformers

In [ ]:
import hashlib, json, os, re
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none (this will be slow)')

In [ ]:
LOCAL = False   # True = small local test

if LOCAL:
    WORK = 'data/selector/gemma_test'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/selector'
for f in ['paragraphs_v3.csv', 'candidates_links10.jsonl']:
    assert os.path.exists(f'{WORK}/{f}'), f'Missing {f} in {WORK}'
print('working folder:', WORK)

## The selector code (`build_training_set.py`, `selector.py`)

In [ ]:
MIN_CHARS = 50
SKIP_SECTIONS = {"References", "See also", "External links", "Further reading", "Notes", "Bibliography",
                 "Sources", "Citations", "Footnotes", "Explanatory notes", "Works cited"}

In [ ]:
def split_paragraphs(content):
    """[(section path, paragraph), ...] following the rules in the module docstring."""
    out, path, skip = [], [], False
    for line in content.split("\n"):
        line = line.strip()
        heading = re.match(r"^(=+)\s*(.*?)\s*=+$", line)
        if heading:
            level = len(heading.group(1)) - 1          # "== X ==" is level 1
            path = path[:level - 1] + [heading.group(2)]
            if level == 1:
                skip = heading.group(2) in SKIP_SECTIONS
            continue
        if not skip and len(line) >= MIN_CHARS:
            out.append((" > ".join(path) or "(intro)", line))
    return out

In [ ]:
E5_MODEL = "intfloat/multilingual-e5-large"
GEMMA_MODEL = "google/embeddinggemma-2"
MODEL = GEMMA_MODEL
PARAGRAPH_THRESHOLD = 0.7
ACCEPT_THRESHOLD = 0.30
REJECT_THRESHOLD = 0.10
MIN_RELATED_PARAGRAPHS = 7

In [ ]:
def load_encoder(model_name=MODEL, device=None):
    """The embedding model. EmbeddingGemma 2 is loaded text-only (no image/audio parts), in float32 as its card says."""
    from sentence_transformers import SentenceTransformer
    if model_name == GEMMA_MODEL:
        import torch
        encoder = SentenceTransformer(model_name, device=device,
                                      config_kwargs={"vision_config": None, "audio_config": None},
                                      model_kwargs={"torch_dtype": torch.float32})
        encoder.max_seq_length = 2048
        return encoder
    return SentenceTransformer(model_name, device=device)

In [ ]:
class Selector:
    def __init__(self, encoder, model_name=MODEL, paragraph_threshold=PARAGRAPH_THRESHOLD,
                 accept_threshold=ACCEPT_THRESHOLD, reject_threshold=REJECT_THRESHOLD,
                 min_related_paragraphs=MIN_RELATED_PARAGRAPHS, clf=None):
        self.encoder = encoder              # a sentence_transformers.SentenceTransformer (load_encoder)
        self.model_name = model_name
        self.paragraph_threshold = paragraph_threshold
        self.accept_threshold = accept_threshold
        self.reject_threshold = reject_threshold
        self.min_related_paragraphs = min_related_paragraphs
        self.clf = clf

    # ---------------------------------------------------------------- vectors
    def embed(self, texts, batch_size=32, show_progress_bar=False):
        """Paragraph vectors, with each model's own prompt: E5 expects "query: ", Gemma its classification prompt."""
        if self.model_name == E5_MODEL:
            return self.encoder.encode(["query: " + t for t in texts], batch_size=batch_size,
                                       normalize_embeddings=True, show_progress_bar=show_progress_bar)
        return self.encoder.encode(texts, prompt_name="Classification", batch_size=batch_size,
                                   normalize_embeddings=True, show_progress_bar=show_progress_bar)

    # ---------------------------------------------------------------- training
    def fit(self, paragraphs, labels, vectors=None):
        """Logistic regression on the paragraph vectors. class_weight="balanced" evens out the
        different number of paragraphs in the two groups."""
        x = self.embed(paragraphs, show_progress_bar=True) if vectors is None else vectors
        self.clf = LogisticRegression(C=10, max_iter=2000, class_weight="balanced").fit(x, labels)
        return self

    # ---------------------------------------------------------------- scoring
    def paragraph_scores(self, paragraphs, vectors=None):
        """Probability (0..1) that each paragraph is related."""
        if vectors is None:
            if not paragraphs:
                return np.array([])
            vectors = self.embed(paragraphs)
        return self.clf.predict_proba(vectors)[:, 1]

    def page_score(self, paragraph_probs, paragraph_threshold=None):
        """Share of the page's paragraphs that count as related (0..1)."""
        t = self.paragraph_threshold if paragraph_threshold is None else paragraph_threshold
        return float(np.mean(np.asarray(paragraph_probs) >= t)) if len(paragraph_probs) else 0.0

    def decision(self, score, related=0):
        """"accept", "review" or "reject" for a page score (and its number of related paragraphs)."""
        if score >= self.accept_threshold or (self.min_related_paragraphs and related >= self.min_related_paragraphs):
            return "accept"
        return "reject" if score < self.reject_threshold else "review"

    def score_page(self, text):
        """Everything for one page's raw text: {"score", "decision", "paragraphs", "related_paragraphs"}."""
        paragraphs = [p for _, p in split_paragraphs(text)]
        probs = self.paragraph_scores(paragraphs)
        score = self.page_score(probs)
        related = int((probs >= self.paragraph_threshold).sum())
        return {"score": round(score, 3), "decision": self.decision(score, related), "paragraphs": len(paragraphs),
                "related_paragraphs": related}

    # ---------------------------------------------------------------- save / load
    def save(self, folder):
        """The classifier and the settings; the embedding model itself is downloaded by name when loading."""
        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        joblib.dump(self.clf, folder / "selector_logreg.joblib")
        (folder / "selector_config.json").write_text(json.dumps({
            "model_name": self.model_name, "paragraph_threshold": self.paragraph_threshold,
            "accept_threshold": self.accept_threshold, "reject_threshold": self.reject_threshold,
            "min_related_paragraphs": self.min_related_paragraphs}, indent=2),
            encoding="utf-8")

    @classmethod
    def load(cls, folder, encoder=None):
        folder = Path(folder)
        config = json.loads((folder / "selector_config.json").read_text(encoding="utf-8"))
        if encoder is None:
            encoder = load_encoder(config["model_name"])
        return cls(encoder, clf=joblib.load(folder / "selector_logreg.joblib"), **config)

In [ ]:
def page_counts(probs, para_page, n_pages, t):
    """(related paragraphs, share) per page at paragraph threshold t."""
    related = np.bincount(para_page, weights=probs >= t, minlength=n_pages).astype(int)
    total = np.bincount(para_page, minlength=n_pages)
    return related, np.divide(related, total, out=np.zeros(n_pages), where=total > 0)

def cached_chunks(texts, folder, selector, chunk=5000):
    """Vectors for all texts, computed in chunks saved to `folder` (named like ScoreNewPages.ipynb); chunks already
    there are reused. The embedding model is loaded only if a chunk is missing."""
    folder = Path(folder); folder.mkdir(exist_ok=True)
    parts = []
    for start in range(0, len(texts), chunk):
        path = folder / f'{start:07d}_{len(texts)}.npy'
        if not path.exists():
            if selector.encoder is None:
                selector.encoder = load_encoder(selector.model_name, device=DEVICE)
            np.save(path, selector.embed(texts[start:start + chunk]))
            print('  computed vectors', min(start + chunk, len(texts)), '/', len(texts), flush=True)
        parts.append(np.load(path))
    return np.vstack(parts)

## 1. Train the final selector (and E5 v3, for the comparison)

In [ ]:
train = pd.read_csv(f'{WORK}/paragraphs_v3.csv', encoding='utf-8-sig')
if LOCAL:
    train = train.groupby('page').head(8)
fingerprint = hashlib.md5('\n'.join(train.text).encode('utf-8')).hexdigest()[:8]
selectors = {'gemma': Selector(None, model_name=GEMMA_MODEL), 'e5': Selector(None, model_name=E5_MODEL)}
for name, cache in [('gemma', f'{WORK}/train_vectors_gemma2_v3_{fingerprint}.npy'),   # from CompareEmbeddings.ipynb
                    ('e5', f'{WORK}/train_vectors_e5_v3_{fingerprint}.npy')]:        # from SelectorV3.ipynb
    sel = selectors[name]
    if not os.path.exists(cache):
        sel.encoder = load_encoder(sel.model_name, device=DEVICE)
        np.save(cache, sel.embed(train.text.tolist(), show_progress_bar=True))
    sel.fit(None, train.label.to_numpy(), vectors=np.load(cache))
selectors['gemma'].save(f'{WORK}/model_gemma')
print('training paragraphs:', train.label.value_counts().to_dict(), '| final selector saved to', f'{WORK}/model_gemma')
print(open(f'{WORK}/model_gemma/selector_config.json').read())

## 2. Score the BFS candidates with both models

In [ ]:
cands = [json.loads(l) for l in open(f'{WORK}/candidates_links10.jsonl', encoding='utf-8')]
if LOCAL:
    cands = cands[:3]
c_page, c_text = [], []
for i, p in enumerate(cands):
    for _, text in split_paragraphs(p['content']):
        c_page.append(i); c_text.append(text)
c_page = np.array(c_page, dtype=int)
print(len(cands), 'candidates,', len(c_text), 'paragraphs')

vectors = {'e5': cached_chunks(c_text, Path(WORK) / 'candidate_vectors_links10', selectors['e5']),   # already in Drive
           'gemma': cached_chunks(c_text, Path(WORK) / 'candidate_vectors_links10_gemma2', selectors['gemma'])}

In [ ]:
out = pd.DataFrame({'title': [p['title'] for p in cands], 'links_from_corpus': [p.get('links_from_corpus') for p in cands],
                    'paragraphs': np.bincount(c_page, minlength=len(cands))})
for name in ['gemma', 'e5']:
    sel = selectors[name]
    related, share = page_counts(sel.paragraph_scores(None, vectors=vectors[name]), c_page, len(cands), PARAGRAPH_THRESHOLD)
    out[f'{name}_score'], out[f'{name}_related'] = np.round(share, 3), related
    out[f'{name}_decision'] = [sel.decision(s, r) for s, r in zip(share, related)]
out.to_csv(f'{WORK}/gemma_candidate_scores.csv', index=False)
pd.crosstab(out.gemma_decision, out.e5_decision, margins=True)

Download `gemma_candidate_scores.csv` and the folder `model_gemma` from `MyDrive/selector`.